# Night Flash — Kaggle

Six cells. Cells 3 to 6 are **byte-identical** to the Colab notebook; only
the data plumbing differs, because Kaggle wipes `/kaggle/working` between
sessions and you attach data as a Dataset instead of mounting Drive.

### Before you start

1. Upload `training_photos/` as a Kaggle Dataset (normal_photos, disposable_photos, validation_photos).
2. Attach it in the notebook settings, on the right-hand side.
3. Accelerator: **GPU T4 x2**. A ~4M model will not saturate one T4, so the
   second GPU is best spent on a second experiment in a copy of this notebook.

### Resuming across sessions

`/kaggle/working` does not survive. After a session, snapshot the run as a
Dataset version, then attach and restore it next time:

```python
from nightflash.kaggle import snapshot, restore
snapshot(s.run_dir)     # end of session N
restore(s.run_dir)      # start of session N+1, after attaching the dataset
s.train(extra_epochs=40)
```

In [ ]:
# ── 1. install ──────────────────────────────────────────────────────
import subprocess, sys, os
from pathlib import Path

PROJECT = Path("/kaggle/working/nightflash-edge")
INPUT = Path("/kaggle/input")

# Find the code: a git URL, an unzipped copy, or your own dataset of the repo.
REPO_URL = os.environ.get("NIGHTFLASH_REPO_URL", "")
if REPO_URL and not PROJECT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT)], check=True)
if not (PROJECT / "pyproject.toml").exists():
    raise RuntimeError(
        "Put the repo at /kaggle/working/nightflash-edge. Either clone it in a "
        "cell above, or upload the repo as a Dataset and unzip it here."
    )
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT)], check=True)
sys.path.insert(0, str(PROJECT / "src"))

import torch
from nightflash import Session
from nightflash.briefing import brief
from nightflash.notebook import show_previews, show_image
from nightflash.kaggle import environment_report, recommended_device

for line in ["kaggle:", *[f"  {k}: {v}" for k, v in environment_report().items()]]:
    print(line)

In [ ]:
# ── 2. data — attached Kaggle Datasets live under /kaggle/input ─────
def find_dataset_dir(must_contain: str) -> Path:
    """Locate the attached dataset that holds the requested subfolder."""
    for candidate in sorted(INPUT.iterdir()):
        if (candidate / must_contain).is_dir():
            return candidate
    raise FileNotFoundError(
        f"No attached dataset contains {must_contain!r}. "
        f"Attach it on the right-hand panel. Saw: {[p.name for p in INPUT.iterdir()]}"
    )

DATASET = find_dataset_dir("normal_photos")
RUNS = Path("/kaggle/working/runs")

SOURCE_DIR = DATASET / "normal_photos"
TARGET_DIR = DATASET / "disposable_photos"
VAL_DIR = DATASET / "validation_photos"
MASK_DIR = None  # set to DATASET/"source_masks" when you have Vision masks

for label, path in [("source", SOURCE_DIR), ("target", TARGET_DIR)]:
    if not path.is_dir():
        raise FileNotFoundError(f"{label} not found: {path}")
    n = sum(1 for p in path.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
    print(f"{label:7s} {n:6d} images")
print("runs root (wiped each session):", RUNS)

In [ ]:
# ── 3. configuration — the only thing you normally edit ─────────────
s = Session(
    arch="edgehat",          # edgehat | swinir | restormer | hybrid
    head="freqsplit",        # freqsplit | lut3d | perpixel | tanh
    name="night",            # run folder suffix: edgehat_freqsplit_<name>
    source_dir=SOURCE_DIR,
    target_dir=TARGET_DIR,
    val_dir=VAL_DIR,
    mask_dir=MASK_DIR,
    runs_root=RUNS,
    image_size=256,
    batch_size=8,
    epochs=50,                # absolute target, not extra epochs
    device=recommended_device("auto"),
    use_amp=True,
    num_workers=2,
    preview_every=250,
    log_every=25,
    halation=True,
    subject_gate=True,
    region_exposure=True,
)
print(s)

In [ ]:
# Pick ONE. Each line is idempotent and safe to re-run.

s.train()                                   # resume, or start fresh
# s.train(extra_epochs=20)                  # one session's budget
# s.infer(f"{SOURCE_DIR}", seeds=3)        # inference only
# s.infer(photo, tile=512)                  # full-resolution detail
# show_previews(s.run_dir, n=6)

# Heads: tanh | perpixel | freqsplit | lut3d
#        lut3d+freqsplit | lut3d+perpixel | tanh+freqsplit

# End of session? Snapshot so the next one can resume:
# from nightflash.kaggle import snapshot
# snapshot(s.run_dir)


In [ ]:
# Pick ONE. Each line is idempotent and safe to re-run.

s.train()                                   # resume, or start fresh
# s.train(extra_epochs=20)                  # one session's budget
# s.infer(f"{SOURCE_DIR}", seeds=3)        # inference only
# s.infer(photo, tile=512)                  # full-resolution detail
# show_previews(s.run_dir, n=6)

# Heads: tanh | perpixel | freqsplit | lut3d
#        lut3d+freqsplit | lut3d+perpixel | tanh+freqsplit

# End of session? Snapshot so the next one can resume:
# from nightflash.kaggle import snapshot
# snapshot(s.run_dir)


In [ ]:
# ── 6. inspect and collect ─────────────────────────────────────────
show_previews(s.run_dir, n=4)
print("\nRun directory:", s.run_dir)
print("State:", s.state().status)

print("To keep this run, Save Version with output, or snapshot it as a Dataset:")
print("  from nightflash.kaggle import snapshot")
print("  snapshot(s.run_dir)")

print("Core ML export should happen on macOS, not here.")
print("Download:", s.latest if s.latest.exists() else "(no checkpoint yet)")